In [1]:
import os
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import cv2
from tqdm import tqdm
import zipfile


/usr/local/lib/python3.10/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.5 (you have 1.4.20). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:

albumentations_transform_for_saving = A.Compose([
    A.Resize(256, 256),
    A.RandomGamma(gamma_limit=(80, 120), p=1.0),
])


In [3]:
def custom_collate(batch):
    images, labels, filenames = zip(*batch)
    return list(images), list(labels), list(filenames)


In [4]:
class BreakHisDataset(Dataset):
    def __init__(self, root_dir, transform=None, for_saving=False):
        self.root_dir = root_dir
        self.transform = transform
        self.for_saving = for_saving
        self.image_paths = []
        self.labels = []
        self.class_names = []

        subclasses = ['adenosis', 'fibroadenoma', 'tubular_adenoma', 'phyllodes_tumor',
                      'ductal_carcinoma', 'lobular_carcinoma', 'mucinous_carcinoma', 'papillary_carcinoma']

        self.class_names = subclasses

        for subclass in subclasses:
            for label_type in ['benign', 'malignant']:
                subclass_path = os.path.join(root_dir, label_type, 'SOB', subclass)
                if not os.path.exists(subclass_path):
                    continue
                for patient_folder in os.listdir(subclass_path):
                    patient_path = os.path.join(subclass_path, patient_folder)
                    for mag_folder in os.listdir(patient_path):
                        mag_path = os.path.join(patient_path, mag_folder)
                        for file in os.listdir(mag_path):
                            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                                self.image_paths.append(os.path.join(mag_path, file))
                                self.labels.append(subclasses.index(subclass))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
        label = self.labels[idx]
        filename = os.path.basename(img_path)
    
        return image, label, filename  
    


In [5]:
def save_gamma_corrected_images(dataset, transform_for_saving, output_dir, batch_size=32):
    dataloader = DataLoader(dataset_for_saving, batch_size=32, shuffle=False, collate_fn=custom_collate, num_workers=2)


    os.makedirs(output_dir, exist_ok=True)
    print("Saving gamma-corrected images...")

    for images, labels, filenames in tqdm(dataloader):
        for i in range(len(images)):
            
            img_path = dataset.image_paths[i]
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            transformed = transform_for_saving(image=image)["image"]

            
            class_name = dataset.class_names[labels[i]]  

            class_folder = os.path.join(output_dir, class_name)
            os.makedirs(class_folder, exist_ok=True)

            save_path = os.path.join(class_folder, filenames[i])
            cv2.imwrite(save_path, cv2.cvtColor(transformed, cv2.COLOR_RGB2BGR))


In [6]:
def zip_folder(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, folder_path)
                zipf.write(file_path, arcname)


In [7]:

input_root = "/kaggle/input/breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides/breast"
output_root = "/kaggle/working/gamma_corrected_images"
zip_output = "/kaggle/working/gamma_corrected_images.zip"


dataset_for_saving = BreakHisDataset(root_dir=input_root, transform=None)
print(len(dataset_for_saving))

save_gamma_corrected_images(dataset_for_saving, albumentations_transform_for_saving, output_root)

zip_folder(output_root, zip_output)

print(f" Gamma corrected images saved and zipped to: {zip_output}")


7909
Saving gamma-corrected images...


100%|██████████| 248/248 [02:50<00:00,  1.45it/s]


✅ Gamma corrected images saved and zipped to: /kaggle/working/gamma_corrected_images.zip
